# Capítulo 6: Construyendo el Primer Modelo (Del Código al Modelo)

> *"Construir un modelo de Machine Learning es como cocinar una receta: necesitas los ingredientes correctos, la secuencia adecuada, y sobre todo, saber cuándo el plato está listo para servir — y cuándo está envenenado."*

Este notebook contiene el pipeline completo de ML, desde datos crudos hasta evaluación. Cada celda sigue la estructura del Mandamiento 3: configuración, ETL con antes/después, modelo, fit, y post-mortem.

## Celda 1: Configuración y Carga de Librerías

### ¿Por qué scikit-learn 1.3+?

- **scikit-learn 1.3** (2023): Mejor soporte para datos categóricos, optimizaciones de rendimiento, corrección de bugs en OneHotEncoder
- **pandas 2.0+**: Tipos nullable nativos (`pd.Int64Dtype()`), mejor manejo de memoria con `pyarrow` backend
- **Python 3.10+**: Structural pattern matching, mejor tipado estático

No usamos versiones más antiguas porque:
- sklearn < 1.2 tiene bugs conocidos en `ColumnTransformer`
- pandas < 2.0 consume 3-5x más memoria con strings
- Python < 3.9 tiene EOL (End of Life) y no recibe patches de seguridad

In [ ]:
# Celda 1: Configuración
# Por qué scikit-learn 1.3+: soporte nativo para categorías, mejoras de rendimiento
# Por qué pandas 2.0+: tipos nullable nativos, mejor manejo de memoria
# Por qué Python 3.10+: EOL en versiones anteriores, security patches

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"scikit-learn: 1.3+ (verificado en importación)")
print("\nConfiguración completada.")

## Celda 2: Carga de Datos

### Dataset: Empleados HR (Reutilizado del Cap 4)

¿Por qué reusar datos? Porque en el mundo real, los datasets no cambian entre capítulos. Es el mismo dataset de rotación de empleados con 1470 registros y 23 features.

**Columnas clave:**
- `Attrition` (target): ¿El empleado renunció? (Yes/No)
- `Age`, `MonthlyIncome`, `DistanceFromHome`: Numéricas
- `Department`, `Gender`, `OverTime`: Categóricas
- `JobSatisfaction`, `WorkLifeBalance`: Ordinales

In [ ]:
# Celda 2: Carga de datos
# Dataset: Empleados HR - predicción de rotación (Attrition)
# Reusar datos es buena práctica: consistencia entre análisis
# 1470 registros, 23 features, target binario (Yes/No)

df = pd.read_csv('../datos/datos_empleados_hr.csv')

print(f"Shape: {df.shape}")
print(f"\nColumnas: {list(df.columns)}")
print(f"\nTarget distribution:")
print(df['Attrition'].value_counts(normalize=True))
print(f"\nTipos de datos:")
print(df.dtypes.value_counts())

## Celda 3: Antes de la Transformación (df.head() ANTES)

### ¿Qué miramos aquí?

1. **Valores categóricos crudos:** "Sales", "Research & Development", "Yes", "No"
2. **Rangos numéricos:** Edad 18-60, Income 1009-20000
3. **Valores faltantes:** ¿Hay NaNs?
4. **Encoding potencial:** Las categóricas necesitan convertirse a números

**Nota ética:** Observar si hay variables protegidas (Gender, Age) que podrían generar sesgo.

In [ ]:
# Celda 3: ANTES de la transformación
# Observar: valores categóricos crudos, rangos numéricos,encoding necesario
# NOTA ÉTICA: Gender y Age son variables protegidas — usar con precaución

print("=== ANTES DE LA TRANSFORMACIÓN ===")
print(f"\nPrimeras 5 filas:")
df.head()

## Celda 4: Transformaciones ETL

### ¿Qué transformamos y por qué?

1. **OneHotEncoder** para categóricas nominales (Department, Gender, OverTime):
   - RandomForest no entiende strings
   - One-Hot crea columnas binarias por categoría
   - `drop='first'` evita multicolinealidad

2. **StandardScaler** para numéricas:
   - Aunque RandomForest no lo necesita estrictamente, es buena práctica
   - Permite comparar importancias entre features con diferentes escalas

3. **ColumnTransformer** para aplicar transformaciones diferentes por tipo de columna:
   - Evita data leakage (aplicar fit en test)
   - Mantiene el pipeline limpio y reproducible

In [ ]:
# Celda 4: Transformaciones ETL
# OneHotEncoder para categóricas: convierte strings a binarios
# StandardScaler para numéricas: normaliza rangos
# ColumnTransformer: aplica transformaciones diferentes por tipo de columna
# IMPORTANTE: fit solo en train, transform en train y test (evita data leakage)

# Definir features por tipo
categorical_features = ['Department', 'Gender', 'OverTime']
numerical_features = ['Age', 'DistanceFromHome', 'MonthlyIncome', 
                      'NumCompaniesWorked', 'PercentSalaryHike',
                      'TotalWorkingYears', 'TrainingTimesLastYear',
                      'YearsAtCompany', 'YearsInCurrentRole',
                      'YearsSinceLastPromotion', 'YearsWithCurrManager']

# Preprocesador
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), 
         categorical_features)
    ]
)

print("Preprocesador definido:")
print(f"  - Features numéricas ({len(numerical_features)}): {numerical_features[:3]}...")
print(f"  - Features categóricas ({len(categorical_features)}): {categorical_features}")

## Celda 5: Después de la Transformación (df.head() DESPUÉS)

### ¿Qué cambió?

1. **Variables categóricas:** De strings a binarios (0/1)
2. **Variables numéricas:** De escalas originales a escalas estandarizadas
3. **Nuevas columnas:** Department_RD, Department_Sales, Gender_Male, OverTime_Yes

**La trampa del encoding:** One-Hot crea `n-1` columnas para `n` categorías (por `drop='first'`). Esto evita multicolinealidad pero dificulta la interpretación.

In [ ]:
# Celda 5: DESPUÉS de la transformación
# Comparar con Celda 3: mismas filas, pero features transformadas
# Categóricas → binarias, Numéricas → estandarizadas

# Primero separamos X e y
X = df.drop('Attrition', axis=1)
y = df['Attrition'].map({'Yes': 1, 'No': 0})

# Aplicar preprocesamiento
X_processed = preprocessor.fit_transform(X)

# Obtener nombres de columnas transformadas
cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)
all_feature_names = list(numerical_features) + list(cat_feature_names)

# Crear DataFrame para visualizar
X_processed_df = pd.DataFrame(X_processed, columns=all_feature_names)

print("=== DESPUÉS DE LA TRANSFORMACIÓN ===")
print(f"\nShape original: {X.shape}")
print(f"Shape transformado: {X_processed_df.shape}")
print(f"\nPrimeras 5 filas transformadas:")
X_processed_df.head()

## Celda 6: Split Train/Test

### ¿Por qué 80/20?

Es una convención, no una ley:
- **Train (80%):** Donde el modelo aprende patrones
- **Test (20%):** Donde evaluamos si realmente aprendió

**La trampa del leakage:** Si hacemos `fit_transform` en todo el dataset, el modelo "ve" los datos de test durante el entrenamiento. Por eso usamos `fit_transform` solo en train y `transform` en test.

**¿Por qué stratify?** Para mantener la proporción de clases (Yes/No) en train y test. Si no lo hacemos, podríamos tener un test sin ninguna instancia de la clase minoritaria.

In [ ]:
# Celda 6: Split train/test
# 80% train (el modelo aprende aquí), 20% test (evaluamos aquí)
# stratify=y: mantiene proporción de clases en ambos sets
# random_state=42: reproducibilidad — mismo split siempre

X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # mantengo proporción de clases
)

print(f"Train: {X_train.shape[0]} muestras ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test: {X_test.shape[0]} muestras ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nDistribución train:")
print(pd.Series(y_train).value_counts(normalize=True))
print(f"\nDistribución test:")
print(pd.Series(y_test).value_counts(normalize=True))

## Celda 7: Modelo - Random Forest

### ¿Por qué RandomForest?

Es el **SUV del ML**: no es el más rápido, no es el más elegante, pero funciona en casi todo terreno.

**Hiperparámetros clave y por qué los elegimos:**
- `n_estimators=200`: Más árboles = más estabilidad, pero más tiempo
- `max_depth=10`: Controlar overfitea (¡la trampa más común!)
- `min_samples_split=10`: Para dividir, necesito al menos 10 datos
- `min_samples_leaf=5`: Cada hoja debe tener al menos 5 datos
- `class_weight='balanced'`: Ajustar por clases desbalanceadas (16% Yes, 84% No)
- `random_state=42`: Reproducibilidad

In [ ]:
# Celda 7: Modelo Random Forest
# 200 árboles: más que 100 (default), más estable sin ser excesivo
# max_depth=10: controlar overfitea — ¡la trampa más común!
# class_weight='balanced': clases desbalanceadas (16% Yes vs 84% No)
# random_state=42: reproducibilidad

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1  # usar todos los cores disponibles
)

print("Modelo configurado:")
print(f"  - Tipo: {model.__class__.__name__}")
print(f"  - Número de árboles: {model.n_estimators}")
print(f"  - Profundidad máxima: {model.max_depth}")
print(f"  - Muestras mínimas para split: {model.min_samples_split}")
print(f"  - Muestras mínimas en hoja: {model.min_samples_leaf}")
print(f"  - class_weight: {model.class_weight}")

## Celda 8: Entrenamiento (model.fit())

### ¿Qué hace model.fit()?

1. **Bootstrap sampling:** Cada árbol se entrena con una muestra con reemplazo (~63% de los datos únicos)
2. **Feature sampling:** En cada nodo, solo considera `sqrt(n_features)` features al azar
3. **Construcción de árboles:** Cada árbol crece hasta `max_depth` o hasta que no pueda dividir más
4. **Votación mayoritaria:** La predicción final es el consenso de todos los árboles

**La trampa del tiempo:** Con 200 árboles y 1470 muestras, esto debería tomar < 5 segundos. Si toma minutos, algo está mal (datos corruptos,太多的 features, o hardware limitado).

In [ ]:
# Celda 8: Entrenamiento
# model.fit() ejecuta el algoritmo de Random Forest:
# 1. Bootstrap sampling (muestreo con reemplazo)
# 2. Feature sampling (sqrt(n_features) por nodo)
# 3. Construcción de árboles hasta max_depth
# 4. Votación mayoritaria para predicciones

import time

start_time = time.time()
model.fit(X_train, y_train)
end_time = time.time()

print(f"Entrenamiento completado en {end_time - start_time:.2f} segundos")
print(f"Número de árboles entrenados: {model.n_estimators}")
print(f"Feature importances disponibles: {model.feature_importances_ is not None}")

## Celda 9: Predicciones

### ¿Qué estamos prediciendo?

Para cada empleado en el set de test, el modelo decide:
- **1 (Yes):** El empleado probablemente renunciará
- **0 (No):** El empleado probablemente se quedará

**La pregunta ética:** ¿Qué hacemos con esta predicción? Si ofrecemos bonos de retención solo a los que el modelo predice como "en riesgo", ¿estamos siendo justos? ¿O estamos perpetuando sesgos del pasado?

In [ ]:
# Celda 9: Predicciones
# model.predict(): voto mayoritario de 200 árboles
# model.predict_proba(): probabilidad de cada clase (más informativo)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]  # probabilidad de "Yes"

print(f"Predicciones generadas: {len(y_pred)}")
print(f"\nDistribución de predicciones:")
print(pd.Series(y_pred).value_counts())
print(f"\nProbabilidades promedio por clase:")
print(f"  P(No): {y_proba[y_pred == 0].mean():.3f}")
print(f"  P(Yes): {y_proba[y_pred == 1].mean():.3f}")

## Celda 10: Reporte de Clasificación

### Interpretación de métricas

| Métrica | Valor | Significado |
|---------|-------|-------------|
| **Accuracy** | ?% | ¿Cuántos acertó en total? |
| **Precision (Yes)** | ?% | De los que predije "Yes", ¿cuántos realmente se fueron? |
| **Recall (Yes)** | ?% | De los que realmente se fueron, ¿cuántos detecté? |
| **F1-Score (Yes)** | ?% | Balance entre precision y recall |

**Para rotación:** Nos importa el **recall**. Es mejor identificar falsamente a alguien como "en riesgo" (FP) que perder a un empleado valioso sin haberlo detectado (FN).

In [ ]:
# Celda 10: Reporte de clasificación
# Precision: de los que predije "Yes", ¿cuántos realmente se fueron?
# Recall: de los que realmente se fueron, ¿cuántos detecté?
# F1: balance entre precision y recall
# PARA ROTACIÓN: nos importa RECALL — es mejor un FP que un FN

print("=== REPORTE DE CLASIFICACIÓN ===")
print(classification_report(y_test, y_pred, target_names=['No', 'Yes']))

print(f"\nAccuracy global: {accuracy_score(y_test, y_pred):.3f}")

## Celda 11: Matriz de Confusión

### La verdad desnuda

```
                  Predicho: No    Predicho: Sí
Real: No          TN              FP
Real: Sí          FN              TP
```

- **TN (True Negatives):** Predijimos que no se van y no se fueron. Bien.
- **FP (False Positives):** Predijimos que se van pero no se fueron. Gastamos un bono innecesario. Mal menor.
- **FN (False Negatives):** Predijimos que no se van pero sí se fueron. **El peor escenario.** Perdimos a alguien sin saberlo.
- **TP (True Positives):** Predijimos que se van y se fueron. Detectamos el problema. Bien.

In [ ]:
# Celda 11: Matriz de confusión
# TN: predije No y es No (bien)
# FP: predije Yes pero es No (gasté bono innecesario — mal menor)
# FN: predije No pero es Yes (perdí al empleado — EL PEOR ESCENARIO)
# TP: predije Yes y es Yes (detecté el problema — bien)

cm = confusion_matrix(y_test, y_pred)

print("=== MATRIZ DE CONFUSIÓN ===")
print(f"\n{'':>15} Predicho: No  Predicho: Yes")
print(f"{'Real: No':>15} {cm[0][0]:>12}  {cm[0][1]:>13}")
print(f"{'Real: Yes':>15} {cm[1][0]:>12}  {cm[1][1]:>13}")

print(f"\nInterpretación:")
print(f"  - True Negatives (TN): {cm[0][0]} (predije No correctamente)")
print(f"  - False Positives (FP): {cm[0][1]} (predije Yes, pero era No)")
print(f"  - False Negatives (FN): {cm[1][0]} (predije No, pero era Yes) ← EL PROBLEMA")
print(f"  - True Positives (TP): {cm[1][1]} (predije Yes correctamente)")

print(f"\nTasa de falsos negativos: {cm[1][0]/(cm[1][0]+cm[1][1]):.3f}")
print(f"→ {cm[1][0]} empleados en riesgo no fueron detectados")

## Celda 12: Post-Mortem - ¿Qué hizo mal?

### Autocrítica del modelo

Todo modelo tiene problemas. La diferencia entre un junior y un senior es que el senior los busca activamente.

**Preguntas del post-mortem:**
1. ¿El modelo tiene sentido para el negocio?
2. ¿Qué features son más importantes? ¿Son éticas?
3. ¿Qué errores son más costosos? (FN vs FP)
4. ¿Qué variables proxy estamos usando?
5. ¿El modelo perpetúa sesgos del pasado?

In [ ]:
# Celda 12: Post-Mortem
# Análisis de lo que salió mal y qué aprender

print("=== POST-MORTEM: ¿Qué hizo mal? ===")

# 1. Feature importance
print("\n--- Feature Importance (Top 10) ---")
feature_importance = pd.Series(
    model.feature_importances_, 
    index=all_feature_names
).sort_values(ascending=False)

print(feature_importance.head(10))

# 2. Análisis de errores
print("\n--- Análisis de Errores ---")
errors = X_test[y_pred != y_test]
error_true = y_test[y_pred != y_test]
error_pred = y_pred[y_pred != y_test]

fn_count = ((error_true == 1) & (error_pred == 0)).sum()
fp_count = ((error_true == 0) & (error_pred == 1)).sum()

print(f"Falsos Negativos (FN): {fn_count}")
print(f"  → Empleados que se fueron y NO detectamos")
print(f"  → Costo: pérdida de talento, costos de reclutamiento")
print(f"\nFalsos Positivos (FP): {fp_count}")
print(f"  → Empleados que NO se fueron y SÍ detectamos como riesgo")
print(f"  → Costo: bonos de retención innecesarios")

# 3. Sesgos potenciales
print("\n--- Sesgos Potenciales ---")
print("Variables protegidas en el dataset: Gender, Age")
print("Feature más importante:", feature_importance.index[0])
print("\nSi Gender o Age aparecen como top features, debemos investigar")
print("si el modelo está perpetuando sesgos de género o edad.")

# 4. Lecciones aprendidas
print("\n--- Lecciones Aprendidas ---")
print("1. El accuracy NO es la métrica principal para problemas desbalanceados")
print("2. Los falsos negativos son más costosos que los falsos positivos")
print("3. La ética NO es opcional — debe estar en cada decisión")
print("4. El post-mortem es más valioso que el modelo")
print("5. El 80% del tiempo es ETL, el 20% es el modelo")

## Resumen del Capítulo 6

### Lo que construimos
1. **Pipeline completo:** Datos → ETL → Modelo → Evaluación
2. **Transformaciones:** OneHotEncoder + StandardScaler con ColumnTransformer
3. **Modelo:** Random Forest con hiperparámetros justificados
4. **Evaluación:** Classification report + Matriz de confusión
5. **Post-Mortem:** Análisis de errores y sesgos

### Lo que aprendimos
- El 80% del tiempo es ETL, no el modelo
- La ética debe estar incrustada, no al final
- El post-mortem es más valioso que las métricas
- Un modelo "bueno" puede ser perjudicial si perpetúa sesgos

### Cita final
> *Los modelos son espejos. Reflejan lo que les damos. Si les damos datos con sesgos, nos devuelven predicciones con sesgos. La pregunta no es "¿es ético el modelo?" sino "¿somos éticos al construirlo?"*